In [9]:
import os
import pandas as pd
import duckdb as db
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import joblib
from sklearn.metrics import accuracy_score, classification_report, f1_score, roc_auc_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.naive_bayes import MultinomialNB

In [10]:
load_dotenv()

db_path = os.getenv("DATABASE_PATH")
conn = db.connect(db_path)

df = conn.sql("""Select title, text, is_fake FROM processed_news
""").df()

In [11]:
df['full_text'] = df['title'] + " " + df['text']

In [12]:
X = df['full_text']
y = df['is_fake']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [13]:
custom_stop_words = list(ENGLISH_STOP_WORDS) + ['reuters', 'said', 'via', 'twitter', 'video', 'watch', 'breaking', 'shared']

In [14]:
models_to_train = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "XGBoost": xgb.XGBClassifier(n_jobs=-1, random_state=42)
}

In [18]:
results = []
best_accuracy = 0
best_model = ""
best_pipeline = None

for name, model in models_to_train.items():
    pipe = Pipeline([
        ("vectorizer", TfidfVectorizer(stop_words=custom_stop_words)),
        ("model", model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    probs= pipe.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    roc_auc = roc_auc_score(y_test, probs)
    print(f'{name} accuracy {acc}, f1 {f1} roc_auc_score {roc_auc}')
    results.append({
        "Model": name,
        "Accuracy": acc,
        "F1-Score": f1,
        "Roc_Auc_Score": roc_auc
    })

    if acc > best_accuracy:
        best_accuracy = acc
        best_model = name
        best_pipeline = pipe
        
results_df = pd.DataFrame(results).sort_values(by="Accuracy", ascending=False)

Logistic Regression accuracy 0.9753196930946292, f1 0.9727977448907682 roc_auc_score 0.9964984979445557
Naive Bayes accuracy 0.9295396419437341, f1 0.9208219571777554 roc_auc_score 0.9808197401707601
XGBoost accuracy 0.9815856777493606, f1 0.979746835443038 roc_auc_score 0.9984420127542953


In [46]:
veta_1 = ["The government announced new economic reforms today to support local businesses."]
veta_2 = ["The government announced new economic reforms today to support local businesses, Reuters reports."]

print("Prediction without Reuters:", pipe.predict(veta_1))
print("Prediction with Reuters:", pipe.predict(veta_2))

Predpoveď bez Reuters: [1]
Predpoveď s Reuters: [1]
